# B2 — Heatmap: Hour vs Roll Day

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (wk, wm) in zip(axes, WINDOWS_META.items()):
    rk = wm['result_key']
    ts = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'timeseries.parquet')
    ts.index = pd.to_datetime(ts.index)
    ts['roll_day'] = ts.index.normalize().map(
        {pd.Timestamp(d): i for i, d in enumerate(wm['days'])})
    ts['hour_utc'] = ts.index.hour
    ts = ts.dropna(subset=['roll_day'])
    pivot = ts.pivot_table(values='dev', index='hour_utc', columns='roll_day', aggfunc='mean')
    absmax = np.abs(pivot.values).max()
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdBu_r',
                   vmin=-absmax, vmax=absmax, origin='lower')
    ax.set_xticks(range(len(wm['day_labels'])))
    ax.set_xticklabels(wm['day_labels'], fontsize=7)
    ax.set_xlabel('Roll Day')
    ax.set_ylabel('Hour UTC')
    ax.set_title(wk)
    plt.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle('Mean Deviation Heatmap: Hour vs Roll Day', fontsize=13)
save_fig(fig, 'B2_heatmap_hour_vs_rollday.png')
